In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import imageio

import src.DetectingRegionInfo as DetectingRegionInfo
import src.DopplerInfo as DopplerInfo
import src.LinesGenerator as LinesGenerator
import src.WAndDopplerGenerator as WAndDopplerGenerator
import src.FeaturesAndLabelsGenerator as FeaturesAndLabelsGenerator
import src.Config as Config
import src.Config as cf
import src.TrajectoryDataset as TrajectoryDataset
import src.UavModel as UavModel
from src.kan.LBFGS import LBFGS


###############################################################################
# 1. INITIAL SETUP
###############################################################################
# Choose which option to run:
#   OPTION 1: Use 50 seeds for the line generation part, and plot 50 single-trajectory plots.
#   OPTION 2: Use a single selected seed, then plot multiple partial-plots for that one seed.
#   OPTION 3: Same as OPTION 2, but also animate those partial plots into a GIF.
MODE = 3  # (1, 2, or 3)

# For Option 1:
SEED_LIST = list(range(1, 51))  # or any 50 distinct seeds
OUTPUT_FOLDER_OPTION1 = "./tmp/option1_seeds"

# For Option 2 and 3:
SELECTED_SEED = 47  # Any integer you like
OUTPUT_FOLDER_OPTION2 = "./tmp/option2_single_seed"
OUTPUT_FOLDER_OPTION2_GIF = "./tmp/option2_single_seed_gif"

# Make sure the output folders exist
os.makedirs(OUTPUT_FOLDER_OPTION1, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_OPTION2, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_OPTION2_GIF, exist_ok=True)

# Create info structs from config
detecting_region_info = DetectingRegionInfo.DetectingRegionInfo(
    Config.transmittor_position,
    Config.receiver_position_1,
    Config.receiver_position_2,
    Config.receiver_position_3
)
doppler_info = DopplerInfo.DopplerInfo(Config.c, Config.fc, Config.v)


###############################################################################
# 2. VISUALIZATION FUNCTIONS
###############################################################################
def visualize_lines(detecting_region_info, lines, title="Lines", save_path=None):
    """
    Plots the lines in the detecting region. If save_path is given, saves to disk.
    Otherwise, simply shows the plot.
    """
    _, ax = plt.subplots()
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 150)
    ax.add_patch(Polygon(
        [detecting_region_info.v1, detecting_region_info.v2,
         detecting_region_info.v4, detecting_region_info.v3],
        fill=False
    ))

    # Plot the lines
    for line in lines:
        ax.plot([p[0] for p in line], [p[1] for p in line], "r")

    plt.title(title)

    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()


def labels_to_coords(detecting_region_info, labels):
    """
    The label is [distance, angle], where distance is from
    the transmitter to the point, and angle is the angle
    between x-axis and the line from transmitter to point.
    This function converts that label pair to [x, y].
    """
    ref = detecting_region_info.transmittor_position
    coords = []

    for label in labels:
        distance = label[0]
        angle = label[1]
        x = ref[0] + distance * np.cos(angle)
        y = ref[1] + distance * np.sin(angle)
        coords.append([x, y])

    return np.array(coords)


def plot_result(
    detecting_region_info,
    loss_to_show,
    trues,
    predicteds,
    folder_name,
    plot_name
):
    """
    Plots GT vs. predicted in X-Y space. Optionally saves to disk.
    """
    plt.figure(figsize=(10, 6))
    plt.scatter(
        trues[:, 0],
        trues[:, 1],
        label="True",
        marker="o",
        s=30,
        alpha=0.7,
    )
    plt.scatter(
        predicteds[:, 0],
        predicteds[:, 1],
        label="Predicted",
        marker="x",
        s=30,
        alpha=0.7,
    )
    plt.xlabel("X-coordinate")
    plt.ylabel("Y-coordinate")
    plt.legend()
    plt.title("True vs. Predicted")
    plt.grid(True)

    # Draw boundary polygon
    plt.gca().add_patch(Polygon([
        detecting_region_info.v1,
        detecting_region_info.v2,
        detecting_region_info.v4,
        detecting_region_info.v3
    ], fill=False))

    plt.text(
        0,
        0,
        f"Loss: {loss_to_show:.4f}",
        ha="left",
        va="bottom",
        transform=plt.gca().transAxes,
    )

    os.makedirs(folder_name, exist_ok=True)
    test_result_name = os.path.join(folder_name, plot_name)
    plt.savefig(test_result_name)
    plt.close()


###############################################################################
# 3. PREDICTION EVALUATION HELPER
###############################################################################
def generate_and_evaluate_one_trajectory(seed=None, folder_name=".", prefix=""):
    """
    1) Generate lines (with a given seed or None).
    2) Re-parse lines into smaller segments.
    3) Generate doppler, features, labels.
    4) Visualize lines.
    5) Evaluate using a pretrained model.
    6) Create a single plot with all points. Return (loss, predicted_xy, true_xy).
    """
    # -------------------------------------------------------------------------
    # Step A: Generate lines
    # -------------------------------------------------------------------------
    lines_a, lines_b = LinesGenerator.generateLines(
        detecting_region_info=detecting_region_info,
        a_b_distance=Config.a_b_distance,
        num_of_lines_to_generate=1,  # single line
        step_count_per_line=Config.test_set_long_num + (Config.step_count_per_line - 1),
        length_per_step=Config.length_per_step,
        angle_change_limit_per_step=Config.angle_change_limit_per_step,
        seed=seed
    )

    # lines_a, lines_b -> shapes: (1, X, 2)
    # Re-parse them into multiple smaller lines
    lines_a, lines_b = LinesGenerator.reparseSingleLineAsLines(
        line_a=lines_a[0],
        line_b=lines_b[0],
        num_of_lines_to_generate=Config.test_set_long_num,
        step_count_per_line=Config.step_count_per_line,
        seed=seed  # does nothing currently, but for consistency
    )

    # -------------------------------------------------------------------------
    # Step B: Generate W/doppler, features/labels
    # -------------------------------------------------------------------------
    w, doppler = WAndDopplerGenerator.generateWAndDoppler(
        detecting_region_info=detecting_region_info,
        doppler_info=DopplerInfo.DopplerInfo(Config.c, Config.fc, Config.v),
        lines_a=lines_a,
        lines_b=lines_b
    )

    test_long_features, test_long_coor_labels = FeaturesAndLabelsGenerator.generateFeaturesAndLabels(
        detecting_region_info=detecting_region_info,
        lines_a=lines_a,
        w=w,
        doppler=doppler,
        num_of_lines_to_generate=Config.test_set_long_num,
        step_count_per_line=Config.step_count_per_line
    )

    # Visualization: lines_a is shape (test_set_long_num, step_count_per_line, 2)
    combined_lines_a = [la for la in lines_a]  # list of np arrays

    # Choose filenames
    if prefix:
        title_for_plot = f"{prefix}_seed_{seed}"
        path_for_plot = os.path.join(folder_name, f"{prefix}_seed_{seed}_lines.png")
    else:
        title_for_plot = f"seed_{seed}"
        path_for_plot = os.path.join(folder_name, f"seed_{seed}_lines.png")

    visualize_lines(
        detecting_region_info=detecting_region_info,
        lines=combined_lines_a,
        title=title_for_plot,
        save_path=path_for_plot
    )

    # -------------------------------------------------------------------------
    # Step C: Evaluate with pretrained model
    # -------------------------------------------------------------------------
    test_long_dataset = TrajectoryDataset.TrajectoryDataset(
        test_long_features,
        test_long_coor_labels
    )

    # For simplicity, use batch_size = test_set_long_num
    test_long_loader = DataLoader(
        test_long_dataset, batch_size=Config.test_set_long_num, shuffle=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    using_model = UavModel.UavModel()
    model = using_model.to(device)

    pretrained_model_path = "./keymodel/model.pth"
    if not os.path.exists(pretrained_model_path):
        raise FileNotFoundError("No pretrained model found!")
    model.load_state_dict(
        torch.load(pretrained_model_path, map_location=device)
    )
    model.eval()

    criterion = nn.MSELoss()
    total_test_long_loss = 0
    predicted_labels = []
    true_labels = []

    with torch.no_grad():
        for features, labels in test_long_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            test_long_loss = criterion(outputs, labels)
            total_test_long_loss += test_long_loss.item()
            predicted_labels.append(outputs.cpu().numpy())
            true_labels.append(labels.cpu().numpy())

    predicted_labels = np.concatenate(predicted_labels, axis=0)
    true_labels = np.concatenate(true_labels, axis=0)

    long_loss = total_test_long_loss / len(test_long_loader)

    # If dealing directly with [x,y]:
    true_xy = true_labels
    predicted_xy = predicted_labels

    # Single final plot of the entire trajectory
    single_plot_name = f"{prefix}_seed_{seed}_all.png" if prefix else f"seed_{seed}_all.png"
    plot_result(
        detecting_region_info=detecting_region_info,
        loss_to_show=long_loss,
        trues=true_xy,
        predicteds=predicted_xy,
        folder_name=folder_name,
        plot_name=single_plot_name
    )

    return long_loss, predicted_xy, true_xy


def generate_and_evaluate_multiple_partial_plots(true_xy, predicted_xy, folder_name=".", prefix="option2_partial"):
    """
    Create multiple plots, each with a growing subset of points.
    Returns the list of filenames generated (so we can build a GIF later).
    """
    filenames = []
    num_plots = min(len(true_xy), Config.test_set_long_num)

    for i in range(1, num_plots + 1):
        partial_true_xy = true_xy[:i, :]
        partial_predicted_xy = predicted_xy[:i, :]

        partial_loss = np.mean((partial_true_xy - partial_predicted_xy)**2)

        # We generate a filename like "option2_partial_1.png", "option2_partial_2.png", etc.
        partial_plot_name = f"{prefix}_{i}.png"

        plot_result(
            detecting_region_info=detecting_region_info,
            loss_to_show=partial_loss,
            trues=partial_true_xy,
            predicteds=partial_predicted_xy,
            folder_name=folder_name,
            plot_name=partial_plot_name
        )
        filenames.append(partial_plot_name)
        print(f"Saved partial plot {i}/{num_plots} with partial loss {partial_loss:.4f}")

    return filenames


def make_gif_from_pngs(
    src_folder,
    pattern_prefix="option2_partial_",
    output_path="./tmp/option2_single_seed_gif/merged.gif",
    duration=0.5,
    skip=1
):
    """
    Creates a GIF from PNG files in src_folder that match [pattern_prefix]*.png.

    Args:
        src_folder      : Folder to look for PNGs.
        pattern_prefix  : Prefix for PNG files to consider.
        output_path     : Where to save the resulting GIF.
        duration        : Seconds each frame is displayed (smaller => faster).
        skip            : If skip=2, only take every 2nd frame.
    """
    # Collect matching .png files
    all_files = [
        f for f in os.listdir(src_folder)
        if f.startswith(pattern_prefix) and f.endswith(".png")
    ]
    if not all_files:
        print(f"No PNG files found in {src_folder} with prefix '{pattern_prefix}'!")
        return

    # Sort them by numeric portion in the filename, e.g. "option2_partial_5.png" -> 5
    def extract_num(filename):
        return int(filename.replace(pattern_prefix, "").replace(".png", ""))

    all_files.sort(key=extract_num)

    # Apply skipping
    selected_files = all_files[::skip]

    # Build the GIF
    with imageio.get_writer(output_path, mode='I', duration=duration) as writer:
        for file_name in selected_files:
            full_path = os.path.join(src_folder, file_name)
            image = imageio.imread(full_path)
            writer.append_data(image)

    print(f"GIF created at {output_path}, using {len(selected_files)} frames.")


###############################################################################
# 4. MAIN LOGIC: CHOOSE OPTION 1, 2, OR 3
###############################################################################
def main():
    if MODE == 1:
        ########################
        # OPTION 1
        ########################
        # Use 50 seeds, single-trajectory for each seed
        for seed in SEED_LIST:
            print(f"\n=== OPTION 1: Evaluating seed={seed} ===")
            long_loss, predicted_xy, true_xy = generate_and_evaluate_one_trajectory(
                seed=seed,
                folder_name=OUTPUT_FOLDER_OPTION1,
                prefix="option1"
            )
            print(f"[ Seed={seed} ] => Single Plot Loss: {long_loss:.4f}")

    elif MODE == 2:
        ########################
        # OPTION 2
        ########################
        # Use one selected seed, produce partial-step plots
        print(f"\n=== OPTION 2: Using SELECTED_SEED={SELECTED_SEED} ===")
        long_loss, predicted_xy, true_xy = generate_and_evaluate_one_trajectory(
            seed=SELECTED_SEED,
            folder_name=OUTPUT_FOLDER_OPTION2,
            prefix="option2"
        )
        print(f"[ Seed={SELECTED_SEED} ] => Single Plot Loss: {long_loss:.4f}")

        generate_and_evaluate_multiple_partial_plots(
            true_xy=true_xy,
            predicted_xy=predicted_xy,
            folder_name=OUTPUT_FOLDER_OPTION2,
            prefix="option2_partial"
        )

    else:
        ########################
        # OPTION 3
        ########################
        # Same as OPTION 2, but then also build a GIF
        print(f"\n=== OPTION 3: Using SELECTED_SEED={SELECTED_SEED}, then building GIF ===")
        long_loss, predicted_xy, true_xy = generate_and_evaluate_one_trajectory(
            seed=SELECTED_SEED,
            folder_name=OUTPUT_FOLDER_OPTION2,
            prefix="option2"
        )
        print(f"[ Seed={SELECTED_SEED} ] => Single Plot Loss: {long_loss:.4f}")

        # 1) Generate partial-step PNGs
        generate_and_evaluate_multiple_partial_plots(
            true_xy=true_xy,
            predicted_xy=predicted_xy,
            folder_name=OUTPUT_FOLDER_OPTION2,
            prefix="option2_partial"
        )

        # 2) Turn the partial-step PNGs into a GIF
        gif_file_name = f"option2_seed_{SELECTED_SEED}.gif"
        gif_full_path = os.path.join(OUTPUT_FOLDER_OPTION2_GIF, gif_file_name)

        # The partial plots have the prefix "option2_partial_"
        # e.g. "option2_partial_1.png", "option2_partial_2.png", etc.
        make_gif_from_pngs(
            src_folder=OUTPUT_FOLDER_OPTION2,
            pattern_prefix="option2_partial_",
            output_path=gif_full_path,
            duration=0.5,  # seconds per frame
            skip=1         # take every frame
        )


if __name__ == "__main__":
    main()